In [1]:
!pip install transformers datasets evaluate captum scikit-learn pandas -q


# Attribution Analysis
IG attributions vs human rationales across all tokenizers.


In [2]:
import pandas as pd
import numpy as np
import time
import sys
import os
sys.path.append(os.path.abspath('../src'))
from attribution import LRAttribution, RobertaAttribution, BertAttribution, AlbertAttribution
from evaluation import aggregate_word_attributions, compute_token_f1
from transformers import AutoTokenizer


In [3]:
adv_df = pd.read_pickle('../data/processed/adv_test.pkl')
row = adv_df.iloc[0]
original_text = row['original_text']
perturbed_text = row['perturbed_text']


In [6]:
attributions = {
    'lr_char': LRAttribution(model_dir='../models/lr_char'),
    'lr_word': LRAttribution(model_dir='../models/lr_word'),
    'roberta': RobertaAttribution(model_dir='../models/roberta'),
    'bert': BertAttribution(model_dir='../models/bert'),
    'albert': AlbertAttribution(model_dir='../models/albert')
}


FileNotFoundError: Model not found at ../models/lr_char/lr_model.joblib

In [22]:
for name, attr_model in attributions.items():
    orig_tokens, orig_scores = attr_model.get_attribution(original_text)
    adv_tokens, adv_scores = attr_model.get_attribution(perturbed_text)
    print(f'\n--- {name.upper()} ---')
    print('Original:', orig_tokens)
    print('Adversarial:', adv_tokens)


NameError: name 'attributions' is not defined

# Token-Level F1
Compare IG attribution alignment with human rationales before and after adversarial attack.
space_injection is excluded from Token-F1 because it breaks word-level rationale alignment.


In [23]:
test_df = pd.read_pickle('../data/processed/test.pkl')
toxic_test = test_df[(test_df['label'] == 1) & (test_df['rationale'].apply(lambda r: sum(r) > 0))].copy()
print(f'Toxic samples with rationales: {len(toxic_test)}')


Toxic samples with rationales: 822


In [24]:
model_configs = {
    'RoBERTa': (RobertaAttribution, '../models/roberta'),
    'BERT': (BertAttribution, '../models/bert'),
    'ALBERT': (AlbertAttribution, '../models/albert'),
}

all_results = {}


NameError: name 'RobertaAttribution' is not defined

In [25]:
for model_name, (attr_class, model_dir) in model_configs.items():
    print(f'\n{"="*60}')
    print(f'{model_name}')
    print(f'{"="*60}')
    
    attr_model = attr_class(model_dir=model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    
    orig_results = []
    adv_results = {a: [] for a in ['char_insertion', 'leetspeak', 'homoglyph_swap']}
    
    start = time.time()
    
    for idx, (_, row) in enumerate(toxic_test.iterrows()):
        if idx % 50 == 0:
            print(f'  {idx}/{len(toxic_test)} ({time.time()-start:.0f}s)')
        
        try:
            _, scores = attr_model.get_attribution(row['text'], target_class=1)
            word_attrs = aggregate_word_attributions(tokenizer, row['text'], scores)
            r = compute_token_f1(word_attrs, row['rationale'])
            if r: orig_results.append(r)
        except:
            continue
        
        for _, adv_row in adv_df[adv_df['post_id'] == row['post_id']].iterrows():
            if adv_row['attack_type'] == 'space_injection':
                continue
            try:
                _, adv_scores = attr_model.get_attribution(adv_row['perturbed_text'], target_class=1)
                adv_word_attrs = aggregate_word_attributions(tokenizer, adv_row['perturbed_text'], adv_scores)
                r = compute_token_f1(adv_word_attrs, row['rationale'])
                if r: adv_results[adv_row['attack_type']].append(r)
            except:
                continue
    
    print(f'  Done: {len(orig_results)} orig + {sum(len(v) for v in adv_results.values())} adv ({time.time()-start:.0f}s)')
    all_results[model_name] = {'original': orig_results, 'adversarial': adv_results}


NameError: name 'model_configs' is not defined

## Results


In [26]:
for model_name in ['RoBERTa', 'BERT', 'ALBERT']:
    data = all_results[model_name]
    orig = data['original']
    
    orig_f1 = np.mean([r['f1'] for r in orig])
    orig_p = np.mean([r['precision'] for r in orig])
    orig_r = np.mean([r['recall'] for r in orig])
    
    all_adv = [r for v in data['adversarial'].values() for r in v]
    adv_f1 = np.mean([r['f1'] for r in all_adv])
    adv_p = np.mean([r['precision'] for r in all_adv])
    adv_r = np.mean([r['recall'] for r in all_adv])
    
    print(f'\n{model_name}')
    print(f'  Original  - P: {orig_p:.3f}  R: {orig_r:.3f}  F1: {orig_f1:.3f}  (n={len(orig)})')
    print(f'  Adversarial - P: {adv_p:.3f}  R: {adv_r:.3f}  F1: {adv_f1:.3f}  (n={len(all_adv)})')
    print(f'  Delta F1: {adv_f1 - orig_f1:+.3f}')
    
    for attack in ['char_insertion', 'leetspeak', 'homoglyph_swap']:
        results = data['adversarial'][attack]
        if results:
            print(f'    {attack}: F1={np.mean([r["f1"] for r in results]):.3f} (n={len(results)})')


NameError: name 'all_results' is not defined

In [27]:
import json
save_data = {}
for m, data in all_results.items():
    save_data[m] = {'original': data['original'], 'adversarial': data['adversarial']}
with open('../results/token_f1_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print('Saved to results/token_f1_results.json')


NameError: name 'all_results' is not defined